# 🧱 Module 2: Configuration & Tuning
## "Fix the LEGO Data Warehouse — Live"

**Duration:** 45 minutes | **Level:** 300–400

---

### Scenario

The LEGO manufacturing analytics team ran their initial data pipeline with **every Spark and Delta optimization disabled** — no auto-compaction, no optimize-write, no adaptive query execution, no V-Order, no deletion vectors. The result? Thousands of tiny files, full table scans on every query, and painfully slow DML operations.

**Your mission:** Fix each table live, one optimization at a time, and measure the impact.

### Lab Pattern

Every exercise follows the same steps:

| Step | What you do |
|------|------------|
| 🐌 **Benchmark** | Run a query and capture the baseline time/metric |
| 🔍 **Diagnose** | Inspect table metadata to prove the root cause |
| 🔧 **Fix** | Apply the optimization |
| 🚀 **Re-benchmark** | Run the same test and compare against the baseline |

### Exercises

| # | Optimization | Table | What's broken |
|---|-------------|-------|----------------|
| 1 | `OPTIMIZE` (compaction) | `manufacturing_event` | Thousands of tiny files from streaming |
| 2 | Optimize Write | `inventory_transaction` | Every commit adds many small files instead of one |
| 3 | Liquid Clustering | `inventory_transaction` | Selective queries scan every file |
| 4 | Deletion Vectors | `web_order_line` | DELETE/UPDATE rewrites entire files |

---

In [ ]:
# ============================================================
# SETUP — Helpers
# ============================================================
import time
from pyspark.sql import DataFrame, functions as F
from delta.tables import DeltaTable

ORIG_SCHEMA = "bronze"        # original unoptimized tables (never modified)
FAST_SCHEMA = "bronze_lab"       # shallow clones we'll fix during the lab

benchmarks = {}  # reset on cell re-run — clears all prior state


def get_table_metrics(table_name):
    """Return file count, total size, and avg file size for a Delta table."""
    detail = spark.sql(f"DESCRIBE DETAIL {FAST_SCHEMA}.{table_name}").collect()[0]
    num_files = detail["numFiles"]
    size_bytes = detail["sizeInBytes"]
    avg_kb = round(size_bytes / num_files / 1024, 1) if num_files > 0 else 0
    return {"num_files": num_files, "size_mb": round(size_bytes / 1048576, 2), "avg_file_kb": avg_kb}


def show_metrics(table_name, label=""):
    m = get_table_metrics(table_name)
    tag = f" ({label})" if label else ""
    print(f"   {FAST_SCHEMA}.{table_name}{tag}:  {m['num_files']:>6,} files  |  {m['size_mb']:>8.1f} MB  |  avg {m['avg_file_kb']:>8.1f} KB/file")
    return m



class _BenchmarkProxy:
    """
    DataFrame proxy that times the next terminal action.

    Non-terminal ops (.filter, .select, .groupBy, etc.) pass through and
    return a new proxy so you can chain freely. The timer starts when a
    terminal action is called (.collect, .count, .show, etc.).
    """

    TERMINAL_ACTIONS = {
        "collect", "count", "show", "head", "first",
        "take", "toPandas", "printSchema", "explain", "display",
    }

    def __init__(self, df: DataFrame, scenario: str, state: str):
        self._df = df
        self._scenario = scenario
        self._state = state

    def _record(self, elapsed_ms: float):
        """Store timing and print comparison when multiple states exist."""
        benchmarks.setdefault(self._scenario, {})[self._state] = elapsed_ms
        states = benchmarks[self._scenario]

        print(f"\u23f1\ufe0f  {self._scenario} [{self._state}]: {elapsed_ms:.2f} ms")

        if len(states) > 1:
            baseline_key = next(iter(states))
            baseline_ms = states[baseline_key]
            best_ms = min(states.values())
            W = 58  # inner width
            print()
            print(f"  \u250c{'\u2500' * W}\u2510")
            title = f"\033[1m{self._scenario}\033[0m"
            title_pad = W - 2 - len(self._scenario)
            print(f"  \u2502  {title}{' ' * title_pad}\u2502")
            print(f"  \u251c{'\u2500' * W}\u2524")
            print(f"  \u2502  {'State':<28}{'Time (ms)':>12}{'Factor':>14}  \u2502")
            print(f"  \u251c{'\u2500' * W}\u2524")
            for s, ms in states.items():
                ratio = baseline_ms / max(ms, 0.001)
                if s == baseline_key:
                    visible_tag = "baseline"
                    tag = visible_tag
                elif ms <= best_ms:
                    visible_tag = f"{ratio:.1f}x faster"
                    tag = f"\033[1;32m{visible_tag}\033[0m"
                else:
                    visible_tag = f"{ratio:.1f}x"
                    tag = f"\033[1;34m{visible_tag}\033[0m"
                pad = 14 - len(visible_tag)
                print(f"  \u2502  {s:<28}{ms:>12.2f}{' ' * pad}{tag}  \u2502")
            print(f"  \u2514{'\u2500' * W}\u2518")

    def __getattr__(self, name):
        attr = getattr(self._df, name)

        if name in self.TERMINAL_ACTIONS and callable(attr):
            def timed(*args, **kwargs):
                spark = self._df.sparkSession
                spark.catalog.clearCache()
                spark.sparkContext.setJobDescription(f"{self._scenario} [{self._state}]")

                start = time.time()
                result = attr(*args, **kwargs)
                elapsed_ms = (time.time() - start) * 1000

                spark.sparkContext.setJobDescription(None)
                self._record(elapsed_ms)
                return result
            return timed

        if callable(attr):
            def passthrough(*args, **kwargs):
                result = attr(*args, **kwargs)
                if isinstance(result, DataFrame):
                    return _BenchmarkProxy(result, self._scenario, self._state)
                return result
            return passthrough

        return attr


def _benchmark(self: DataFrame, scenario: str, state: str):
    """
    Start a timed benchmark. Chain operations, then call a terminal action.

    Examples:
        df.benchmark("Small Files", "before").count()
        df.benchmark("Small Files", "after").count()
        df.benchmark("Clustering", "off").filter(...).collect()
    """
    return _BenchmarkProxy(self, scenario, state)


# Attach the benchmark method to DataFrame
DataFrame.benchmark = _benchmark

print("\u2705 Setup complete — helpers loaded")
print(f"   Schemas: orig={ORIG_SCHEMA}, lab={FAST_SCHEMA}")


In [ ]:
# ============================================================
# SETUP — Shallow clone tables into lab schema
# ============================================================
# We clone the unoptimized tables so the originals stay untouched.
# This lets you re-run the lab without re-seeding data.
# Shallow clones reference the same Parquet files — instant, zero copy cost.

LAB_TABLES = ["manufacturing_event", "inventory_transaction", "web_order_line"]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FAST_SCHEMA}")

for table in LAB_TABLES:
    spark.sql(f"DROP TABLE IF EXISTS {FAST_SCHEMA}.{table}")
    spark.sql(f"""
        CREATE TABLE {FAST_SCHEMA}.{table}
        SHALLOW CLONE {ORIG_SCHEMA}.{table}
    """)
    print(f"  ✅ {FAST_SCHEMA}.{table} ← shallow clone of {ORIG_SCHEMA}.{table}")

# All exercises operate on FAST_SCHEMA
print(f"\n📌 All exercises will use schema: {FAST_SCHEMA}")
print(f"   Original tables in '{ORIG_SCHEMA}' are preserved for re-runs.")


---

# Warm-up: The Cost of Schema Inference

Before we even run queries, notice how long it takes to simply **discover the schema** from the landing zone. With thousands of small files, Spark must open many of them to infer column types — this is a hidden startup cost every time the pipeline restarts.

**Production best practice:** Define schemas statically so your pipeline starts instantly — no file scanning needed.

---

In [ ]:
# ============================================================
# WARM-UP — Time schema inference on the landing zone
# ============================================================

# The unoptimized pipeline left thousands of small files in the landing zone.
# Schema inference must scan these files to discover the schema.
#
# NOTE: spark.read.json() triggers inference EAGERLY at creation time,
# so we must time the DataFrame construction itself — not an action.

LANDING_TABLE = "manufacturing_event"
landing_path = f"Files/landing/{LANDING_TABLE}"

print(f"🐌 Inferring schema from landing zone: {landing_path}\n")
print("   Spark must open files and read metadata to discover columns + types...\n")

start = time.time()
inferred_df = spark.read.option("multiline", "true").json(landing_path)
time_infer_ms = (time.time() - start) * 1000

benchmarks.setdefault("Schema Inference", {})["inferred (file scan)"] = time_infer_ms
print(f"⏱️  Schema inference: {time_infer_ms:.2f} ms")
inferred_df.printSchema()

In [ ]:
# ============================================================
# WARM-UP — Compare: static schema definition (instant)
# ============================================================
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, BooleanType, DecimalType

# A production pipeline defines the schema once in code — no file scanning needed
static_schema = StructType([
    StructField("EventId", StringType()),
    StructField("Timestamp", TimestampType()),
    StructField("MachineId", StringType()),
    StructField("PartNum", StringType()),
    StructField("ColorId", IntegerType()),
    StructField("MoldTemp", DecimalType(5, 1)),
    StructField("InjectionPressure", DecimalType(6, 1)),
    StructField("CycleTimeMs", IntegerType()),
    StructField("DefectDetected", BooleanType()),
    StructField("DefectType", StringType()),
    StructField("BatchId", StringType()),
])

start = time.time()
static_df = spark.read.schema(static_schema).json(landing_path)
time_static_ms = (time.time() - start) * 1000

benchmarks.setdefault("Schema Inference", {})["static (no scan)"] = time_static_ms
print(f"⏱️  Static schema: {time_static_ms:.2f} ms (no files scanned)")
static_df.printSchema()

# Print comparison
states = benchmarks["Schema Inference"]
if len(states) > 1:
    baseline_ms = list(states.values())[0]
    best_ms = min(states.values())
    W = 58
    print(f"\n  \u250c{'\u2500' * W}\u2510")
    title = f"\033[1mSchema Inference\033[0m"
    title_pad = W - 2 - len("Schema Inference")
    print(f"  \u2502  {title}{' ' * title_pad}\u2502")
    print(f"  \u251c{'\u2500' * W}\u2524")
    print(f"  \u2502  {'State':<28}{'Time (ms)':>12}{'Factor':>14}  \u2502")
    print(f"  \u251c{'\u2500' * W}\u2524")
    baseline_key = next(iter(states))
    for s, ms in states.items():
        ratio = baseline_ms / max(ms, 0.001)
        if s == baseline_key:
            visible_tag = "baseline"
            tag = visible_tag
        elif ms <= best_ms:
            visible_tag = f"{ratio:.1f}x faster"
            tag = f"\033[1;32m{visible_tag}\033[0m"
        else:
            visible_tag = f"{ratio:.1f}x"
            tag = f"\033[1;34m{visible_tag}\033[0m"
        pad = 14 - len(visible_tag)
        print(f"  \u2502  {s:<28}{ms:>12.2f}{' ' * pad}{tag}  \u2502")
    print(f"  \u2514{'\u2500' * W}\u2518")

print("\n📝 Takeaway: production pipelines define schemas upfront.")
print("   Inference is convenient for exploration but adds startup latency,")
print("   especially when scanning thousands of small files.")

---

# Exercise 1: Fix the Small Files Problem

**Table:** `manufacturing_event` — high-frequency IoT telemetry from the injection molding floor

**What's wrong:** The unoptimized pipeline wrote data via Spark Structured Streaming with no auto-compaction and no optimize-write. Every micro-batch created a new tiny Parquet file. The result: thousands of files, each just a few KB.

**Why it matters:**
- Each file requires a separate Spark task → scheduling overhead dominates actual compute
- File metadata (Parquet footer, Delta stats) is disproportionately large vs. data
- The Delta transaction log bloats with thousands of `AddFile` entries

**Fix:** [`OPTIMIZE`](https://learn.microsoft.com/fabric/data-engineering/delta-optimization-and-v-order?tabs=sparksql#optimize) — compacts small files into ~128 MB target files.

---

In [ ]:
# ============================================================
# 1️⃣ BENCHMARK — Capture baseline query time
# ============================================================

print("🐌 Running baseline query on manufacturing_event...\n")

part_query = spark.sql(f"SELECT part_num, COUNT(*) AS total_events FROM {FAST_SCHEMA}.manufacturing_event GROUP BY part_num") \
    .benchmark("OPTIMIZE (compaction)", "before").collect()

In [ ]:
# ============================================================
# 1️⃣ DIAGNOSE — Prove the root cause is small files
# ============================================================

print("🔍 Table diagnostics:\n")
metrics_1_before = show_metrics("manufacturing_event", "before")

if metrics_1_before["avg_file_kb"] < 1024:
    ratio = round(131072 / max(metrics_1_before["avg_file_kb"], 1))
    print(f"\n   ⚠️  Average file is {metrics_1_before['avg_file_kb']:.0f} KB — optimal target is ~128 MB (131,072 KB)")
    print(f"   ⚠️  Files are {ratio:,}× smaller than they should be")

print("\n📋 DESCRIBE DETAIL:")
display(spark.sql(f"DESCRIBE DETAIL {FAST_SCHEMA}.manufacturing_event").select("format", "numFiles", "sizeInBytes"))

In [ ]:
# ============================================================
# 1️⃣ FIX — Run OPTIMIZE to compact small files
# ============================================================

print("🔧 Running OPTIMIZE {FAST_SCHEMA}.manufacturing_event...\n")
start = time.time()
result = spark.sql(f"OPTIMIZE {FAST_SCHEMA}.manufacturing_event")
print(f"   Completed in {round(time.time() - start, 1)}s")
display(result)

In [ ]:
# ============================================================
# 1️⃣ RE-BENCHMARK — Same query, compare against baseline
# ============================================================

print("🚀 Running same query after OPTIMIZE...\n")

part_query_fast = spark.sql(f"SELECT part_num, COUNT(*) AS total_events FROM {FAST_SCHEMA}.manufacturing_event GROUP BY part_num") \
    .benchmark("OPTIMIZE (compaction)", "after").collect()

metrics_1_after = show_metrics("manufacturing_event", "after")

### 💡 What Just Happened?

`OPTIMIZE` rewrote all those tiny files into a smaller number of properly-sized files (~128 MB each). This is a **one-time reactive compaction** — it doesn't prevent small files from appearing again on the next write.

> 📝 **Note:** `OPTIMIZE` creates new files but keeps old files for time travel. Run `VACUUM` to reclaim storage (default retention: 7 days).

---

---

# Exercise 2: Optimize Write

**Table:** `inventory_transaction` — part movements across production lines

**What's wrong:** Look at the Delta transaction log: every streaming commit added many small files when the data could have fit in one. Without **optimize write** (bin-packing at write time), each micro-batch creates a file-per-partition, regardless of how small the data is.

**Why it matters:**
- `OPTIMIZE` (Exercise 1) is **reactive** — you run it *after* the damage
- Optimize write is **proactive** — it bin-packs data *during* the write
- Without it, you're in an endless cycle: write small files → run OPTIMIZE → write small files again

**Approach:**
1. Analyze the Delta log to see the file-per-commit pattern from the original pipeline
2. Replicate the problem with a test write
3. Enable optimize write and repeat — see the difference

---

In [ ]:
# ============================================================
# 2️⃣ DIAGNOSE — Analyze the Delta log: files added per commit
# ============================================================

print("🔍 Analyzing Delta transaction log for inventory_transaction...\n")

# DESCRIBE HISTORY shows each commit version with operation metrics
history = spark.sql(f"DESCRIBE HISTORY {FAST_SCHEMA}.inventory_transaction")

# Extract files added per commit from operationMetrics
commit_stats = (
    history
    .select(
        F.col("version"),
        F.col("timestamp"),
        F.col("operation"),
        F.col("operationMetrics.numOutputRows").cast("long").alias("rows_written"),
        F.col("operationMetrics.numFiles").cast("int").alias("files_added"),
        F.col("operationMetrics.numAddedFiles").cast("int").alias("files_added_alt"),
    )
    .withColumn("files_added", F.coalesce("files_added", "files_added_alt"))
    .filter("files_added IS NOT NULL AND files_added > 0")
    .orderBy("version")
)

display(commit_stats)

print("\n📊 Files added per commit (from the unoptimized streaming pipeline):")
stats_rows = commit_stats.collect()
for r in stats_rows:
    files = r["files_added"]
    rows = r["rows_written"] or 0
    bar = "█" * min(files, 60)
    print(f"   v{r['version']:<4}  {files:>4} files  ({rows:>8,} rows)  {bar}")

In [ ]:
# ============================================================
# 2️⃣ DIAGNOSE — Plot actual vs. ideal file growth
# ============================================================

print("📊 Cumulative file growth: actual vs. ideal (1 file per commit)\n")

# Build cumulative file counts
cum_actual = 0
cum_ideal = 0
print(f"   {'Commit':<10} {'Actual (cum)':>15} {'Ideal (cum)':>15} {'Overhead':>10}")
print(f"   {'-' * 50}")
for r in stats_rows:
    cum_actual += r["files_added"]
    cum_ideal += 1
    overhead = f"{cum_actual / max(cum_ideal, 1):.1f}x"
    print(f"   v{r['version']:<8} {cum_actual:>15,} {cum_ideal:>15,} {overhead:>10}")

total_commits = len(stats_rows)
total_files_created = cum_actual
print(f"\n   ⚠️  {total_commits} commits created {total_files_created:,} files")
print(f"   ⚠️  Ideal: {total_commits} commits should create ~{total_commits} files")
print(f"   ⚠️  Overhead: {total_files_created / max(total_commits, 1):.1f}x more files than necessary")

In [ ]:
# ============================================================
# 2️⃣ BENCHMARK — Replicate: write WITHOUT optimize write
# ============================================================

# Start from a clean compacted state
spark.sql(f"OPTIMIZE {FAST_SCHEMA}.inventory_transaction")
files_before = get_table_metrics("inventory_transaction")["num_files"]

# Simulate a streaming micro-batch by appending data
# Use repartition to mimic how streaming creates multiple partitions
BATCH_ROWS = 5000

print(f"🐌 Appending {BATCH_ROWS:,} rows WITHOUT optimize write (repartitioned to 8 files)...\n")

batch = spark.table(f"{FAST_SCHEMA}.inventory_transaction").limit(BATCH_ROWS).repartition(8)
batch.write.format("delta").mode("append").saveAsTable(f"{FAST_SCHEMA}.inventory_transaction")

files_after_bad = get_table_metrics("inventory_transaction")["num_files"]
new_files_without_ow = files_after_bad - files_before
print(f"   Files before append: {files_before:,}")
print(f"   Files after append:  {files_after_bad:,}")
print(f"   ⚠️  New files created: {new_files_without_ow}")

In [ ]:
# ============================================================
# 2️⃣ FIX — Enable optimize write
# ============================================================

print("🔧 Enabling optimize write on inventory_transaction...\n")
spark.sql("""
    ALTER TABLE {FAST_SCHEMA}.inventory_transaction SET TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact' = 'true'
    )
""")

props = spark.sql(f"SHOW TBLPROPERTIES {FAST_SCHEMA}.inventory_transaction").filter(
    "key LIKE '%autoOptimize%'"
)
display(props)
print("✅ Optimize write + auto compact enabled")

In [ ]:
# ============================================================
# 2️⃣ RE-BENCHMARK — Same append, now with optimize write
# ============================================================

# Clean baseline
spark.sql(f"OPTIMIZE {FAST_SCHEMA}.inventory_transaction")
files_before_good = get_table_metrics("inventory_transaction")["num_files"]

print(f"🚀 Appending {BATCH_ROWS:,} rows WITH optimize write (same repartition to 8)...\n")

batch = spark.table(f"{FAST_SCHEMA}.inventory_transaction").limit(BATCH_ROWS).repartition(8)
batch.write.format("delta").mode("append").saveAsTable(f"{FAST_SCHEMA}.inventory_transaction")

files_after_good = get_table_metrics("inventory_transaction")["num_files"]
new_files_with_ow = files_after_good - files_before_good

print(f"   Files before append: {files_before_good:,}")
print(f"   Files after append:  {files_after_good:,}")
print(f"   New files created:   {new_files_with_ow}")

print(f"\n{'=' * 60}")
print(f"  Exercise 2: Optimize Write")
print(f"{'=' * 60}")
print(f"  {'Metric':<30} {'Without OW':>12} {'With OW':>12}")
print(f"  {'-' * 54}")
print(f"  {'New files per append':<30} {new_files_without_ow:>12} {new_files_with_ow:>12}")
print(f"  {'Overhead vs. ideal (1 file)':<30} {new_files_without_ow:>11}x {'1x':>12}")
print(f"{'=' * 60}")

benchmarks["Exercise 2: Optimize Write"] = {
    "before_s": new_files_without_ow, "after_s": new_files_with_ow,
    "speedup": round(new_files_without_ow / max(new_files_with_ow, 1), 1),
    "before_files": new_files_without_ow, "after_files": new_files_with_ow,
    "note": "files created per append"
}

### 💡 What Just Happened?

Without optimize write, Spark wrote one file per output partition — even though the data was tiny. With optimize write enabled, Spark **bin-packs** the data at write time, coalescing small partitions into properly-sized files.

| Setting | Behavior |
|---------|----------|
| `optimizeWrite = false` | 1 file per partition → 8 files for 5,000 rows |
| `optimizeWrite = true` | Bin-packs into ~1 file (data fits in one ~128 MB target) |
| `autoCompact = true` | Triggers background compaction after writes exceed threshold |

> 📝 **Optimize write is the single most important setting for preventing small files.** It solves the problem at the source — no post-hoc `OPTIMIZE` needed.

---

---

# Exercise 3: Liquid Clustering

**Table:** `inventory_transaction` (continuing from Exercise 2)

**Columns:** `TransactionId`, `Timestamp`, `PartNum`, `ColorId`, `LineId`, `TransactionType`, `Quantity`, `ReferenceId`

**What's wrong:** Even after compaction, data files contain a random mix of all values. When you filter by `PartNum` or `TransactionType`, Spark must scan **every file** because the min/max file-level statistics overlap across all files — nothing can be skipped.

**Why it matters:**
- A query for one specific part number scans 100% of the data
- No files can be pruned via Delta data skipping
- Gets worse as the table grows

**Fix:** [Liquid clustering](https://learn.microsoft.com/fabric/data-engineering/delta-optimization-and-v-order?tabs=sparksql#liquid-clustering) — co-locates related rows in the same files so Delta can skip irrelevant files entirely.

---

In [ ]:
# ============================================================
# 3️⃣ BENCHMARK — Run a selective query (scans everything)
# ============================================================

# Pick a real PartNum from the data
sample_part = spark.sql(f"SELECT PartNum FROM {FAST_SCHEMA}.inventory_transaction LIMIT 1").collect()[0][0]

QUERY_3 = f"""
    SELECT TransactionType, COUNT(*) AS cnt, SUM(Quantity) AS total_qty
    FROM {FAST_SCHEMA}.inventory_transaction
    WHERE PartNum = '{sample_part}'
    GROUP BY TransactionType
"""

# Compact first so file layout is clean
spark.sql(f"OPTIMIZE {FAST_SCHEMA}.inventory_transaction")
metrics_3_before = show_metrics("inventory_transaction", "before clustering")

print(f"\n🐌 Running selective query: WHERE PartNum = '{sample_part}'...\n")

spark.sql(QUERY_3).benchmark("Liquid Clustering", "before (no skipping)").collect()

print("\n🔍 Physical plan — note: ALL files are read, none pruned:")
spark.sql(QUERY_3).explain(True)

In [ ]:
# ============================================================
# 3️⃣ FIX — Apply liquid clustering and re-optimize
# ============================================================

print("🔧 Applying liquid clustering: CLUSTER BY (PartNum, TransactionType)...\n")
spark.sql(f"ALTER TABLE {FAST_SCHEMA}.inventory_transaction CLUSTER BY (PartNum, TransactionType)")
print("✅ Clustering columns set\n")

print("Running OPTIMIZE to rewrite files with clustering layout...")
start = time.time()
spark.sql(f"OPTIMIZE {FAST_SCHEMA}.inventory_transaction")
print(f"   Completed in {round(time.time() - start, 1)}s")

metrics_3_after = show_metrics("inventory_transaction", "after clustering + OPTIMIZE")

In [ ]:
# ============================================================
# 3️⃣ RE-BENCHMARK — Same query, now with clustering
# ============================================================

print(f"🚀 Running same query: WHERE PartNum = '{sample_part}'...\n")

spark.sql(QUERY_3).benchmark("Liquid Clustering", "after (data skipping)").collect()

metrics_3_after = show_metrics("inventory_transaction", "after clustering")

print("\n🔍 Physical plan — check 'files read' vs 'files pruned':")
spark.sql(QUERY_3).explain(True)

print("\n💡 Check Spark UI → SQL tab → Scan node for 'files pruned' —")
print("   with clustering, most files should be skipped for selective filters.")

### 💡 What Just Happened?

Liquid clustering **physically re-organizes data** so rows with similar `PartNum` and `TransactionType` values end up in the same files. Delta's min/max file statistics can then immediately skip files that don't contain matching values.

| Before clustering | After clustering |
|---|---|
| Every file has a random mix of all part numbers | Files contain contiguous ranges of part numbers |
| Scanning `WHERE PartNum = 'X'` reads ALL files | Same query prunes most files via data skipping |

> 🆚 **Liquid clustering vs. partitioning:** Partitioning creates directory-level splits — great for low-cardinality columns, but creates a small files nightmare with high cardinality. Liquid clustering achieves similar data skipping **without** the small files trap, and it's incremental — new data is automatically clustered when `autoCompact` is enabled.

---

---

# Exercise 4: Deletion Vectors

**Table:** `web_order_line` — order line items for LEGO set purchases

**Columns:** `OrderId`, `LineNumber`, `SetNum`, `PartNum`, `ItemName`, `Quantity`, `UnitPrice`, `ExtendedPrice`

**What's wrong:** Without deletion vectors, every `DELETE` or `UPDATE` must **rewrite the entire Parquet files** that contain affected rows. Even deleting a single row triggers a full file rewrite.

**Why it matters:**
- A `DELETE` touching 100 rows across 10 files rewrites all 10 files completely
- Write amplification is enormous — you rewrite GBs to remove KBs
- `MERGE` operations (common in ETL) suffer the same penalty

**Fix:** [Deletion vectors](https://learn.microsoft.com/fabric/data-engineering/delta-optimization-and-v-order?tabs=sparksql#deletion-vectors) — instead of rewriting files, Delta writes a small sidecar file that marks which rows are logically deleted. The original data files stay untouched.

---

In [ ]:
# ============================================================
# 4️⃣ SETUP — Compact the table and pick two test conditions
# ============================================================

print("🔧 Preparing web_order_line (OPTIMIZE to clean baseline)...\n")
spark.sql(f"OPTIMIZE {FAST_SCHEMA}.web_order_line")
metrics_4_before = show_metrics("web_order_line", "baseline")

# Pick two different SetNums for before/after DELETE comparison
set_nums = spark.sql("""
    SELECT SetNum, COUNT(*) as cnt
    FROM {FAST_SCHEMA}.web_order_line
    WHERE SetNum IS NOT NULL
    GROUP BY SetNum
    ORDER BY cnt DESC
    LIMIT 2
""").collect()

set_a, count_a = set_nums[0][0], set_nums[0][1]
set_b, count_b = set_nums[1][0], set_nums[1][1]
print(f"\n   Will DELETE SetNum='{set_a}' ({count_a:,} rows) without DVs")
print(f"   Will DELETE SetNum='{set_b}' ({count_b:,} rows) with DVs")

In [ ]:
# ============================================================
# 4️⃣ BENCHMARK — DELETE without deletion vectors (rewrites files)
# ============================================================

print(f"🐌 DELETE WHERE SetNum = '{set_a}' (without deletion vectors)...\n")

spark.sql(f"DELETE FROM {FAST_SCHEMA}.web_order_line WHERE SetNum = '{set_a}'") \
    .benchmark("Deletion Vectors", "without DVs (file rewrite)").show()

# Check the commit to see how many files were rewritten
history = spark.sql(f"DESCRIBE HISTORY {FAST_SCHEMA}.web_order_line LIMIT 1").select(
    "version", "operation", "operationMetrics"
).collect()[0]
op_metrics = history["operationMetrics"]
files_rewritten_before = int(op_metrics.get("numRemovedFiles", 0))

print(f"   Files rewritten: {files_rewritten_before}")
print(f"   ⚠️  Entire files rewritten just to remove {count_a:,} rows")

In [ ]:
# ============================================================
# 4️⃣ FIX — Enable deletion vectors
# ============================================================

print("🔧 Enabling deletion vectors...\n")
spark.sql("""
    ALTER TABLE {FAST_SCHEMA}.web_order_line SET TBLPROPERTIES (
        'delta.enableDeletionVectors' = 'true'
    )
""")

props = spark.sql(f"SHOW TBLPROPERTIES {FAST_SCHEMA}.web_order_line").filter("key LIKE '%deletionVector%'")
display(props)
print("✅ Deletion vectors enabled")

In [ ]:
# ============================================================
# 4️⃣ RE-BENCHMARK — DELETE with deletion vectors (no file rewrite)
# ============================================================

print(f"🚀 DELETE WHERE SetNum = '{set_b}' (with deletion vectors)...\n")

spark.sql(f"DELETE FROM {FAST_SCHEMA}.web_order_line WHERE SetNum = '{set_b}'") \
    .benchmark("Deletion Vectors", "with DVs (sidecar only)").show()

history = spark.sql(f"DESCRIBE HISTORY {FAST_SCHEMA}.web_order_line LIMIT 1").select(
    "version", "operation", "operationMetrics"
).collect()[0]
op_metrics = history["operationMetrics"]
files_rewritten_after = int(op_metrics.get("numRemovedFiles", 0))

print(f"   Files rewritten: {files_rewritten_after}")
print(f"\n💡 With DVs, Delta wrote a tiny sidecar file marking deleted rows.")
print(f"   The original data files were NOT rewritten — massive write amplification savings.")

---

# 🏆 Summary Dashboard

All four optimizations applied. Here's the full impact across every exercise.

---

In [ ]:
# ============================================================
# SUMMARY — All benchmark results
# ============================================================

print("=" * 62)
print("  🏆  PERFORMANCE IMPACT SUMMARY")
print("=" * 62)

for scenario, states in benchmarks.items():
    if isinstance(states, dict):
        baseline_key = next(iter(states))
        baseline_ms = states[baseline_key]
        best_ms = min(states.values())
        W = 58
        print(f"\n  \u250c{'\u2500' * W}\u2510")
        title = f"\033[1m{scenario}\033[0m"
        title_pad = W - 2 - len(scenario)
        print(f"  \u2502  {title}{' ' * title_pad}\u2502")
        print(f"  \u251c{'\u2500' * W}\u2524")
        print(f"  \u2502  {'State':<28}{'Time (ms)':>12}{'Factor':>14}  \u2502")
        print(f"  \u251c{'\u2500' * W}\u2524")
        for s, ms in states.items():
            ratio = baseline_ms / max(ms, 0.001)
            if s == baseline_key:
                visible_tag = "baseline"
                tag = visible_tag
            elif ms <= best_ms:
                visible_tag = f"{ratio:.1f}x faster"
                tag = f"\033[1;32m{visible_tag}\033[0m"
            else:
                visible_tag = f"{ratio:.1f}x"
                tag = f"\033[1;34m{visible_tag}\033[0m"
            pad = 14 - len(visible_tag)
            print(f"  \u2502  {s:<28}{ms:>12.2f}{' ' * pad}{tag}  \u2502")
        print(f"  \u2514{'\u2500' * W}\u2518")

print(f"\n{'=' * 62}")
print("""
KEY TAKEAWAYS
──────────────
1. OPTIMIZE compacts small files — but it's REACTIVE (run after the damage)
2. Optimize Write bin-packs at write time — PROACTIVE, prevents small files at source
3. Liquid clustering enables data skipping — selective queries skip irrelevant files
4. Deletion vectors eliminate write amplification — DELETEs don't rewrite files
5. The optimized SJD enables ALL of these automatically via ArcFlow's SparkConfigurator —
   proper configuration from the start beats manual fixing every time
""")

---

### 📝 What the Optimized Pipeline Configures Automatically

The `seed_lego_delta_tables` Spark Job Definition uses ArcFlow's `SparkConfigurator` to set these best-practice configs at session startup:

| Setting | Value | What it does |
|---------|-------|-------------|
| `autoCompact.enabled` | `true` | Compacts small files after writes |
| `optimizeWrite.enabled` | `true` | Bin-packs data during writes |
| `targetFileSize.adaptive.enabled` | `true` | Adapts target file size to workload |
| `enableDeletionVectors` | `true` | Marks deleted rows instead of rewriting files |
| `optimize.fast.enabled` | `true` | Faster OPTIMIZE via incremental compaction |
| `parquet.vorder.enabled` | `true` | V-Order encoding for faster reads |
| `parquet.compression.codec` | `zstd` | Better compression ratio than snappy |
| `sql.adaptive.enabled` | `true` | Adaptive Query Execution (AQE) |
| `native.enabled` | `true` | Native Execution Engine |

> 💡 **The best optimization is the one you never have to run manually.**

---